# 6.13 · UMAP / Uniform Manifold Approximation and Projection

> **课程定位 / Where this fits**
> t-SNE(6.12)可视化强但慢、只能可视化、不能 transform 新点、全局结构差。UMAP 同样保局部、画簇漂亮, 但**更快、能 transform 新数据、更好地保留一定全局结构、还能降到任意维度当特征**。它建立在黎曼几何/代数拓扑上, 已成为高维可视化与降维的新主流。
> UMAP preserves local structure like t-SNE but is faster, can transform new data, keeps more global structure, and can reduce to any dimension as features. The modern default.

> 💡 **面试相关 / Interview-relevant**
> - "UMAP vs t-SNE 区别" ★★★★★（速度/transform/全局结构）
> - "UMAP 的核心思想(模糊拓扑图 + 交叉熵)" ★★★★
> - "n_neighbors / min_dist 的作用" ★★★★★
> - "为什么 UMAP 能 transform 新点而 t-SNE 不能" ★★★★

---

## 学习目标 / Learning Objectives
1. UMAP 直觉: 模糊近邻图 + 低维布局(交叉熵)。
2. 与 t-SNE 的对比(速度/transform/全局)。
3. **n_neighbors / min_dist** 两大旋钮。
4. UMAP 作为可 transform 的降维特征。

## 目录 / TOC
1. [UMAP 直觉 + vs t-SNE ⭐](#1)
2. [🔢 数据: Digits + UMAP vs t-SNE ⭐](#2)
3. [n_neighbors / min_dist ⭐](#3)
4. [transform 新数据 ⭐](#4)
5. [小结](#5)


<a id="1"></a>
## 1. UMAP 直觉 + vs t-SNE ⭐ / Intuition & vs t-SNE

UMAP 分两步(和 t-SNE 精神相近但理论不同):
1. **构建高维模糊近邻图**: 对每个点找 `n_neighbors` 个最近邻, 用一个随距离衰减的"模糊隶属度"连边, 得到一张加权图(近似数据所在的流形)。
2. **低维布局**: 在低维随机初始化点, 用**交叉熵**(而非 t-SNE 的 KL)优化, 让低维图的边结构匹配高维图——近邻拉近、非近邻推远(负采样, 像 word2vec)。

**相比 t-SNE 的优势**(面试核心):

| | t-SNE | UMAP |
|---|---|---|
| 速度 | 慢($O(n\log n)$ 优化重) | **快**(尤其大数据) |
| 新数据 | ❌ 不能 transform | ✅ **可 transform**(有可复用映射) |
| 全局结构 | 差(只保局部) | **较好**(保更多全局) |
| 输出维度 | 一般 2-3(可视化) | 任意维(可当降维特征) |
| 目标 | KL 散度 | 交叉熵 + 负采样 |


<a id="2"></a>
## 2. 数据: Digits + UMAP vs t-SNE ⭐ / Head-to-head

复用 **Digits**(6.12 介绍)。同一数据上比 UMAP 与 t-SNE 的布局与速度。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time, warnings
warnings.filterwarnings("ignore")
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
import umap
sns.set_theme(style="whitegrid")
print("umap-learn", umap.__version__)

digits = load_digits(); X, y = digits.data, digits.target

t = time.perf_counter()
Zt = TSNE(2, perplexity=30, init="pca", random_state=0).fit_transform(X)
t_tsne = time.perf_counter()-t

t = time.perf_counter()
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0)
Zu = reducer.fit_transform(X)
t_umap = time.perf_counter()-t

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
axes[0].scatter(Zt[:,0], Zt[:,1], c=y, cmap="tab10", s=8); axes[0].set_title(f"t-SNE ({t_tsne:.1f}s)")
sc = axes[1].scatter(Zu[:,0], Zu[:,1], c=y, cmap="tab10", s=8); axes[1].set_title(f"UMAP ({t_umap:.1f}s)")
plt.colorbar(sc, ax=axes[1], label="digit"); plt.tight_layout(); plt.show()
print("两者都清晰分出 10 个数字簇; UMAP 簇间相对位置更有意义(保更多全局结构)。")
print("⚠️ 墙钟: 此处 UMAP 反而更慢——numba 首次 JIT 编译开销 + random_state 关掉了并行;")
print("   在大数据(数万~数十万点)上 UMAP 的速度优势才显著, 小数据上 t-SNE 可能更快。")


<a id="3"></a>
## 3. n_neighbors / min_dist ⭐ / Two Key Knobs

- **n_neighbors**: 每点考虑的近邻数。**小**→强调极局部细节(更碎、更多小簇); **大**→看更大范围(更全局、更平滑)。类似 t-SNE 的 perplexity。
- **min_dist**: 低维中点能挨多近。**小(0.0–0.1)**→簇内压得很紧实(适合聚类); **大(0.5+)**→点摊得更开(适合看整体分布)。


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for j, nn in enumerate([5, 15, 50]):
    Z = umap.UMAP(n_neighbors=nn, min_dist=0.1, random_state=0).fit_transform(X)
    axes[0,j].scatter(Z[:,0], Z[:,1], c=y, cmap="tab10", s=5)
    axes[0,j].set_title(f"n_neighbors={nn}"); axes[0,j].set_xticks([]); axes[0,j].set_yticks([])
for j, md_ in enumerate([0.0, 0.3, 0.8]):
    Z = umap.UMAP(n_neighbors=15, min_dist=md_, random_state=0).fit_transform(X)
    axes[1,j].scatter(Z[:,0], Z[:,1], c=y, cmap="tab10", s=5)
    axes[1,j].set_title(f"min_dist={md_}"); axes[1,j].set_xticks([]); axes[1,j].set_yticks([])
axes[0,0].set_ylabel("n_neighbors↑\n(局部→全局)"); axes[1,0].set_ylabel("min_dist↑\n(紧实→松散)")
plt.suptitle("UMAP 两大旋钮: n_neighbors(局部vs全局) + min_dist(簇紧实度)")
plt.tight_layout(); plt.show()
print("n_neighbors 小→局部细节多; 大→更全局平滑。min_dist 小→簇紧实; 大→点更分散")


<a id="4"></a>
## 4. transform 新数据 ⭐ / Transforming New Data

t-SNE 不能处理新点; UMAP **学到一个可复用的映射**, 能 `transform` 新样本——所以可以放进真正的 ML pipeline(train 上 fit, test/线上数据 transform), 当降维特征用。


In [ ]:
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit(X_tr)   # 只在训练集 fit
Z_tr = reducer.embedding_
Z_te = reducer.transform(X_te)        # 关键: 对新数据 transform (t-SNE 做不到)

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.scatter(Z_tr[:,0], Z_tr[:,1], c=y_tr, cmap="tab10", s=8, alpha=0.3, label="train(fit)")
ax.scatter(Z_te[:,0], Z_te[:,1], c=y_te, cmap="tab10", s=25, edgecolor="k", label="test(transform)")
ax.legend(); ax.set_title("UMAP 可 transform 新数据: 测试点落到对应簇(t-SNE 无此能力)")
plt.tight_layout(); plt.show()

# 验证: 新点 transform 后用最近邻分类的准确率 / nearest-neighbor on UMAP features
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(5).fit(Z_tr, y_tr)
print(f"在 UMAP 2D 特征上 KNN 分类测试准确率: {knn.score(Z_te, y_te):.3f}")
print("→ UMAP 降到 2D 仍保留足够判别信息, 且能 transform → 可用作生产降维特征")


<a id="5"></a>
## 5. 小结 / Summary

```
UMAP: ①高维模糊近邻图(n_neighbors+衰减隶属度, 近似流形) ②低维交叉熵布局(负采样)
保局部(像 t-SNE)但: 大数据更快 / 可 transform 新点 / 保更多全局结构 / 可降到任意维当特征
n_neighbors: 小→局部细节, 大→全局平滑(类比 perplexity)
min_dist: 小→簇紧实(适合聚类), 大→点松散(看分布)
可放进 pipeline(train fit, 新数据 transform) → 真正的降维特征工具
```

### 💡 面试速查
1. **UMAP = 模糊拓扑近邻图 + 交叉熵低维布局**; t-SNE = 高斯/t + KL
2. **vs t-SNE**: 大数据更快、**可 transform 新点**、保更多全局结构、可任意维(小数据 t-SNE 可能更快, 因 UMAP 有 JIT 开销)
3. **n_neighbors**(局部↔全局) + **min_dist**(簇紧实度)两大旋钮
4. **能 transform** 是因为它学到了可复用映射(t-SNE 每次重算整批)
5. 簇间距离比 t-SNE 稍可信, 但仍以局部结构为主; 可视化首选

### 下一节
**6.14 LDA 作为降维**——5.12 的 LDA 是分类器, 但它本质在找"最能分开类别"的方向, 也是一种(有监督)降维。和 PCA(无监督)正好互补。
